# Does framing change LLM safety? Role x Intent

Backend-parameterized driver for **study 2** -- all logic lives in [mh_safety/role_intent/](mh_safety/role_intent/).
Set `BACKEND` in Setup to run it against any model.

A 2 x 3 factorial: for a fixed situation and model, vary only the user's **role** (victim / perpetrator)
and **intent** (help-seeking / validation / how-to), and measure the **probability of a harmful or
misguiding reply**. Reports a 2 x 3 harm-rate table, main effects, risk ratios + chi-square, and logistic
odds ratios with interaction. The probe prompts are *stimuli*; any harmful content would be the model's
output, which the judge flags. Scenarios are synthetic and non-graphic.

**Judging:** every model's responses are scored by a single fixed judge, `mistralai/Mistral-7B-Instruct-v0.3`
(HuggingFace, GPU) — set on `cfg.judge_llm` — so all models are evaluated the same way.

In [ ]:
# ===== Google Colab setup — run this FIRST on Colab (no-op when running locally) =====
# The local Mistral-7B judge needs a GPU: Runtime > Change runtime type > T4 GPU.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys, subprocess

    REPO_URL = "https://github.com/ana0101/LLM-response-manipulation.git"
    REPO_DIR = "/content/LLM-response-manipulation"

    # 1) code + data (EN_dataset.csv is in the repo; the large data/raw is not needed)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    # 2) dependencies (Colab already ships torch + CUDA; don't reinstall torch)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    subprocess.run('pip install -q "transformers>=4.51.0" accelerate bitsandbytes sentencepiece',
                   shell=True, check=True)

    # 3) HuggingFace login for the gated Mistral judge (accept its licence on HF first).
    #    Add HF_TOKEN in Colab "Secrets" (the key icon), or paste it when prompted.
    from huggingface_hub import login
    from getpass import getpass
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    login(token or getpass("HuggingFace token: "))

    # 4) sentiment lexicon used by the automated metrics
    import nltk; nltk.download("vader_lexicon", quiet=True)

    gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
    print("Colab ready.", gpu or "WARNING: no GPU — set Runtime > Change runtime type > T4 GPU, then re-run.")

## Setup

Pick the model **backend** below; everything else is identical across models. The `mh_safety` pipeline is
backend-agnostic, so this one notebook runs the study on any of Claude / Llama (Ollama) / Gemma / Qwen.

In [ ]:
# ===== pick the model backend =====
BACKEND = "anthropic"        # "anthropic" | "ollama" | "gemma" | "qwen"

# Optional deps by backend:
#   anthropic -> ANTHROPIC_API_KEY (prompted below)
#   ollama    -> a running `ollama` server + `pip install ollama`
#   gemma/qwen-> pip install "transformers>=4.51.0" accelerate bitsandbytes torch   (GPU recommended)

import os, getpass
if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-ant-...): ")

from mh_safety.config import RoleIntentConfig
from mh_safety.llm import LLMClient
from mh_safety.role_intent import pipeline as rp

# EmpathyConfig()/RoleIntentConfig() == anthropic; .ollama()/.gemma()/.qwen() for the others
cfg = RoleIntentConfig() if BACKEND == "anthropic" else getattr(RoleIntentConfig, BACKEND)()
# override anything, e.g.:  cfg = RoleIntentConfig.ollama()
client = LLMClient(cfg.llm)
cfg

## 1. Build role x intent prompts

In [ ]:
prompts = rp.build_prompt_df(cfg)
print(prompts.groupby(["role", "intent"]).size().rename("n"))
prompts[["role", "intent", "prompt"]].head(6)

## 2. Generate + judge

In [ ]:
responses = rp.generate_responses(cfg, prompts, client)
# judged by the shared judge model (Mistral-7B), regardless of which model generated
scored = rp.judge_responses(cfg, responses)
scored[["role", "intent", "harmful", "safety", "behavior", "response"]].head()

## 3. Analyse + report (2x3 harm, main effects, risk ratios, logistic)

In [ ]:
A = rp.analyze(cfg, scored)
rp.print_report(cfg, scored, A)

## 4. Plots + save

In [ ]:
rp.make_plots(cfg, scored, A, show=True)
rp.save_results(cfg, scored, A)

## Notes

* Add scenarios / edit the request grid in [mh_safety/role_intent/scenarios.py](mh_safety/role_intent/scenarios.py).
* Each backend caches to its own dir and writes to its own outputs dir, so runs never collide.
* One-liner equivalent: `rp.run(cfg, show=True)`.
* `theme` is the replication unit (one prompt per theme per cell); cluster by theme for inference.